# CATCHY — Results Overview

Interactive exploration of simulation output.
Run `03_production.py` first to generate data in `output/`.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import h5py

from src.utils import load_config, confinement_parameter, LAMBDA_REF

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# --- Configuration ---
cfg = load_config('../configs/default.yaml')
out_dir = cfg['output']['dir']
rho = cfg['system']['rho_m0']
sigma_list = cfg['enzyme']['sigma_list']
interaction = cfg['enzyme']['interaction']
print(f'Output dir: {out_dir}')
print(f'rho={rho}, lambda={LAMBDA_REF.get(rho, "N/A")}')
print(f'sigma_E values: {sigma_list}')

## MSD(t) — passive vs active

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(sigma_list)))

for mode, ax in zip(['passive', 'active'], axes):
    for i, sigma_E in enumerate(sigma_list):
        C, lam = confinement_parameter(sigma_E, rho)
        label = f'sigma{sigma_E:.1f}_{interaction}_{mode}'
        path = os.path.join(out_dir, f'msd_{label}.npz')
        if not os.path.exists(path):
            continue
        data = np.load(path)
        ax.loglog(data['time'], data['msd'], color=colors[i], label=f'C={C:.2f}')
    ax.set_title(f'{mode} (k_cat={0 if mode=="passive" else cfg["enzyme"]["k_cat"]})')
    ax.set_xlabel('t [LJ units]')
    ax.set_ylabel('MSD [σ²]')
    ax.legend(fontsize=8)
plt.tight_layout()

## D_active / D_passive — enzymatic enhancement

In [ ]:
from analysis.degradation import load_D_table

C_arr, D_act, D_pass = load_D_table(out_dir, sigma_list, rho, interaction)

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(C_arr, D_pass, 'o-', label='Passive', color='steelblue')
ax.semilogy(C_arr, D_act, 's-', label='Active', color='firebrick')

ax2 = ax.twinx()
if len(D_pass) > 0:
    enh = D_act / D_pass
    ax2.semilogy(C_arr, enh, 'D--', color='darkgreen', label='Enhancement')
    ax2.set_ylabel('D_active / D_passive', color='darkgreen')

ax.set_xlabel('C = σ_E / λ')
ax.set_ylabel('Diffusion coefficient D')
ax.set_title('Enzymatic mobility enhancement')
ax.legend(loc='upper right')
plt.tight_layout()

## Bond survival S(t)

In [ ]:
from analysis.degradation import load_survival

fig, ax = plt.subplots(figsize=(6, 4))
for i, sigma_E in enumerate(sigma_list):
    C, _ = confinement_parameter(sigma_E, rho)
    result = load_survival(out_dir, sigma_E, interaction)
    if result is None:
        continue
    t, S = result
    ax.plot(t, S, color=colors[i], label=f'C={C:.2f}')

ax.set_xlabel('t [LJ units]')
ax.set_ylabel('S(t)')
ax.set_title('Bond survival fraction')
ax.legend(fontsize=8)
plt.tight_layout()

## α₂(t) — Non-Gaussian parameter

In [ ]:
from analysis.non_gaussian import non_gaussian_parameter
from src.utils import _unwrap

fig, ax = plt.subplots(figsize=(6, 4))
for i, sigma_E in enumerate(sigma_list[:4]):
    C, _ = confinement_parameter(sigma_E, rho)
    label = f'sigma{sigma_E:.1f}_{interaction}_passive'
    traj_path = os.path.join(out_dir, f'traj_{label}.h5')
    if not os.path.exists(traj_path):
        continue
    with h5py.File(traj_path, 'r') as f:
        epos = f['enzyme_positions'][:]
        L = float(f.attrs['L'])
        t_arr = f['time'][:]
    lag_idx, alpha2 = non_gaussian_parameter(epos, L)
    t_axis = t_arr[:len(lag_idx)]
    ax.semilogx(t_axis, alpha2, color=colors[i], label=f'C={C:.2f}')

ax.set_xlabel('t [LJ units]')
ax.set_ylabel('α₂(t)')
ax.set_title('Non-Gaussian parameter')
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.legend(fontsize=8)
plt.tight_layout()